# User Experience para ciência de dados
Previsão de atrasos em pedidos.

In [ ]:
import pandas as pd

# Importar lógica de preparação de dados existente
from data_preparation import load_and_clean_data

In [87]:
def prepare_data_for_lgbm(df):
    """
    Limpeza e filtragem sugerida para o modelo LightGBM.
    """
    print("\n--- Preparação de Dados para LightGBM ---")
    inicial = len(df)
    
    # 1. Limpeza Crítica: dropna em campos fundamentais
    cols_limpeza = ['dt_despacho_pedido', 'dt_entrega_pedido', 'dt_pagamento_pedido', 'qtd_dias_tat']
    
    # --- OPÇÃO DE IMPUTAÇÃO (COMENTADA) ---
    # Se decidirmos não dropar os nulos de dt_pagamento_pedido (6.4%):
    # df['dt_pagamento_pedido'] = df['dt_pagamento_pedido'].fillna(method='ffill') # ou uma data fixa 'Pendente'
    # df['dias_aprovacao'] = df['dias_aprovacao'].fillna(-1) # Categoria para 'Pendente'
    # --------------------------------------
    
    df = df.dropna(subset=cols_limpeza).copy()
    
    posterior = len(df)
    print(f"Registros antes: {inicial}")
    print(f"Registros após limpeza (dropna): {posterior}")
    print(f"Perda de dados: {1 - (posterior/inicial):.2%}")

    # 2. Tratamento de Outliers (Percentil 99 em qtd_dias_tat)
    limite_99 = df['qtd_dias_tat'].quantile(0.99)
    df = df[df['qtd_dias_tat'] <= limite_99].copy()
    print(f"Outliers removidos (TAT > {limite_99:.1f} dias): {posterior - len(df)}")

    return df

## Limpeza e tratamento de dados

In [96]:
# 1. Carregar e Limpar
input_file = "pedidos_logistica.parquet"
df = load_and_clean_data(input_file, drop_ids=False)

if df is not None:
    df = prepare_data_for_lgbm(df)

Carregando dados de pedidos_logistica.parquet...
Renomeando 'cidade_destinatario_normalizada' para 'cidade_destinatario'
Removendo registros inconsistentes (entrega antecede despacho): 7
Realizando engenharia de features...
Processamento concluído. Formato final: (490210, 23)
Salvando dataset limpo em pedidos_logistica_limpo.parquet...

--- Preparação de Dados para LightGBM ---
Registros antes: 490210
Registros após limpeza (dropna): 458687
Perda de dados: 6.43%
Outliers removidos (TAT > 14.0 dias): 3596


In [97]:
# Manter para ter coerencia com 'trabalho' - Pedro
df = df.drop(
    columns=[
        # 'row_id',
        # 'hr_despacho_pedido',
        'dt_entrega_pedido',
        # 'hr_entrega_pedido',
        'flg_existem_ocorrencias'
    ],
    errors='ignore'
 )

df.head(10)

,id,cod_pedido,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,...,qtd_dias_tat,tp_performance_entrega,cidade_destinatario,hr_despacho_pedido,dias_gastos_cd,dias_transito,dias_atraso_real,dias_ciclo,is_atrasado,dias_restantes_prazo
0,67044,127510252-1,RJ,Transportadora 3,2023-11-27 14:57:25,2023-12-06,2023-11-24,2023-11-24,Capital,Multi,...,5.0,1,RIO DE JANEIRO,14,3.623206,4.209201,-4.167593,7.832407,0,12.0
1,67045,122170353-1,SE,Transportadora 3,2023-07-19 11:41:44,2023-07-27,2023-07-17,2023-07-17,Capital,Mono,...,6.0,1,ARACAJU,11,2.487315,6.127512,-1.385174,8.614826,0,10.0
2,67046,125676313-1,RN,Transportadora 2,2023-11-08 15:44:19,2023-11-20,2023-11-07,2023-11-07,Interior,Mono,...,7.0,1,NATAL,15,1.655775,9.878773,-1.465451,11.534549,0,13.0
3,67047,124809810-1,PA,Transportadora 2,2023-10-16 12:40:54,2023-10-25,2023-10-14,2023-10-14,Reg. Metropolitana,Multi,...,6.0,1,ANANINDEUA,12,2.528403,6.918843,-1.552755,9.447245,0,11.0
5,67049,124134539,MG,Transportadora 1,2023-09-25 08:47:42,2023-10-02,2023-09-24,2023-09-24,Interior,Multi,...,4.0,1,JUIZ DE FORA,8,1.366458,3.065231,-3.568310,4.431690,0,8.0
6,67050,123148738-1,MG,Transportadora 1,2023-08-22 14:13:44,2023-08-30,2023-08-21,2023-08-21,Interior,Multi,...,5.0,1,BAEPENDI,14,1.592870,6.010185,-1.396944,7.603056,0,9.0
7,67051,127085176-1,TO,Transportadora 2,2023-11-24 22:14:31,2023-12-06,2023-11-22,2023-11-22,Capital,Mono,...,6.0,1,PALMAS,22,2.926748,5.557639,-5.515613,8.484387,0,14.0
8,67052,122966837-3,RJ,Transportadora 1,2023-08-16 14:53:27,2023-08-21,2023-08-15,2023-08-15,Reg. Metropolitana,Multi,...,3.0,1,NITEROI,14,1.620451,2.031829,-2.347720,3.652280,0,6.0
9,67053,122721965-1,MG,Transportadora 1,2023-08-07 09:41:30,2023-08-11,2023-08-07,2023-08-06,Interior,Multi,...,3.0,1,POCOS DE CALDAS,9,1.403819,2.162836,-1.433345,2.566655,0,5.0
10,67054,126966272,AC,Transportadora 1,2023-11-23 01:13:53,2023-12-11,2023-11-21,2023-11-21,Capital,Multi,...,9.0,1,RIO BRANCO,1,2.051308,11.681597,-6.267095,13.732905,0,20.0


## Profiling de dados

In [98]:
# profile = ProfileReport(df, title="Profiling Report")
# html = profile.to_html()
# output_file = 'report.html'
# with open(output_file, 'w') as f:
#     f.write(html)

In [99]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
 )

# ---------------------------------------------------------
# Step 3: Split Features (X) and Target (y)
# ---------------------------------------------------------
selected_features = [
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'dt_previsao_entrega_cliente',
    'dt_criacao',
    'dt_pagamento_pedido',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
    # 'dias_gastos_cd',
    # 'dias_restantes_prazo',
]

available_features = [col for col in selected_features if col in df.columns]
missing_features = [col for col in selected_features if col not in df.columns]

if missing_features:
    print('Missing columns (ignored):', missing_features)

X = df[available_features].copy()
y = df['tp_performance_entrega']

# Remove rows with missing target
valid_mask = y.notna()
X = X.loc[valid_mask].copy()
y = y.loc[valid_mask].astype('int32').copy()

# Convert requested date columns to numeric representation (ordinal days)
date_cols = ['dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido']
for col in [c for c in date_cols if c in X.columns]:
    X[col] = pd.to_datetime(X[col], errors='coerce')
    X[col] = X[col].map(lambda x: x.toordinal() if pd.notna(x) else np.nan).astype('float32')

# Encode requested categorical columns as numeric codes
categorical_cols = [
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
]
for col in [c for c in categorical_cols if c in X.columns]:
    X[col] = X[col].astype('category').cat.codes.replace(-1, np.nan).astype('float32')

# LightGBM does not accept object/datetime columns directly
unsupported_cols = X.select_dtypes(include=['object', 'datetime64[ns]', 'datetimetz']).columns
if len(unsupported_cols) > 0:
    print('Dropping unsupported columns:', list(unsupported_cols))
X = X.drop(columns=unsupported_cols, errors='ignore')

print('Training columns:', list(X.columns))

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# Step 4: Initialize and Train the Model
# ---------------------------------------------------------
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    is_unbalance=True
)

# Fit model
model.fit(X_train, y_train)

# ---------------------------------------------------------
# Step 5: Make Predictions (The "Risk Score")
# ---------------------------------------------------------
predictions_binary = model.predict(X_test)
predictions_proba = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Step 6: Evaluate Model Quality
# ---------------------------------------------------------
accuracy = accuracy_score(y_test, predictions_binary)
precision = precision_score(y_test, predictions_binary, zero_division=0)
recall = recall_score(y_test, predictions_binary, zero_division=0)
f1 = f1_score(y_test, predictions_binary, zero_division=0)
roc_auc = roc_auc_score(y_test, predictions_proba)
cm = confusion_matrix(y_test, predictions_binary)

print('=== Model Metrics ===')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC-AUC  : {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(cm)
print('\nClassification Report:')
print(classification_report(y_test, predictions_binary, zero_division=0))

Training columns: ['cidade_destinatario', 'uf', 'grp_transportadora', 'dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido', 'tp_praca', 'des_unidade_negocio', 'des_cd_origem']
[LightGBM] [Info] Number of positive: 351141, number of negative: 12931
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003160 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 798
[LightGBM] [Info] Number of data points in the train set: 364072, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.964482 -> initscore=3.301560
[LightGBM] [Info] Start training from score 3.301560
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

In [100]:
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(10, 6))
# lgb.plot_importance(model, ax=ax)
# plt.tight_layout()
# plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
# plt.close(fig)
# print('Saved: feature_importance.png')

In [101]:
# ---------------------------------------------------------
# Step 7: View the Results
# ---------------------------------------------------------
results_df = X_test.copy()
results_df['probabilidade_atraso'] = predictions_proba
results_df['risco_semaforo'] = pd.cut(
    results_df['probabilidade_atraso'],
    bins=[-0.1, 0.3, 0.7, 1.1],
    labels=['🟢 Verde', '🟡 Amarelo', '🔴 Vermelho']
)
print(results_df[['probabilidade_atraso', 'risco_semaforo']].head(20))

        probabilidade_atraso risco_semaforo
215238              0.906487     🔴 Vermelho
332796              0.534492      🟡 Amarelo
502612              0.818447     🔴 Vermelho
196145              0.573449      🟡 Amarelo
496519              0.642893      🟡 Amarelo
313595              0.709090     🔴 Vermelho
303643              0.623922      🟡 Amarelo
373153              0.520928      🟡 Amarelo
433946              0.733529     🔴 Vermelho
183851              0.200442        🟢 Verde
60913               0.798679     🔴 Vermelho
311994              0.713916     🔴 Vermelho
503263              0.774313     🔴 Vermelho
143350              0.620442      🟡 Amarelo
307862              0.607061      🟡 Amarelo
387118              0.489413      🟡 Amarelo
447206              0.446069      🟡 Amarelo
360264              0.753937     🔴 Vermelho
226825              0.560298      🟡 Amarelo
484414              0.793290     🔴 Vermelho


In [102]:
import joblib

# Build categorical mappings from training data
categorical_mappings = {}
for col in [c for c in categorical_cols if c in df.columns]:
    cats = pd.Series(df.loc[valid_mask, col].astype("string").dropna().unique()).sort_values().tolist()
    categorical_mappings[col] = {v: i for i, v in enumerate(cats)}

bundle = {
    "model": model,
    "selected_features": selected_features,
    "date_cols": ["dt_previsao_entrega_cliente", "dt_criacao", "dt_pagamento_pedido"],
    "categorical_cols": [c for c in categorical_cols if c in selected_features],
    "categorical_mappings": categorical_mappings,
    "threshold": 0.5
}

joblib.dump(bundle, "model_bundle.joblib")
print("Saved model_bundle.joblib")

Saved model_bundle.joblib


In [103]:
df[df['tp_performance_entrega'] == 0]

,id,cod_pedido,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,...,qtd_dias_tat,tp_performance_entrega,cidade_destinatario,hr_despacho_pedido,dias_gastos_cd,dias_transito,dias_atraso_real,dias_ciclo,is_atrasado,dias_restantes_prazo
316043,479646,122901793-1,SP,Transportadora 1,2023-08-14 15:05:53,2023-08-16,2023-08-12,2023-08-12,Capital,Mono,...,4.0,0,SAO PAULO,15,2.629086,2.913819,1.542905,5.542905,1,4.0
316089,479647,126132587-1,SP,Transportadora 1,2023-11-15 19:57:35,2023-11-20,2023-11-13,2023-11-13,Interior,Multi,...,6.0,0,JACAREI,19,2.831655,6.963125,2.794780,9.794780,1,7.0
316316,479836,127372657,SP,Transportadora 5,2023-11-28 01:42:33,2023-11-30,2023-11-24,2023-11-24,Capital,Multi,...,5.0,0,SAO PAULO,1,4.071215,3.489618,1.560833,7.560833,1,6.0
316452,479979,122595633,GO,Transportadora 3,2023-08-04 10:11:17,2023-08-15,2023-08-02,2023-08-02,Interior,Mono,...,14.0,0,SILVANIA,10,2.424502,18.061863,7.486366,20.486366,1,13.0
318043,480408,126274484,SP,Transportadora 5,2023-11-17 08:08:13,2023-11-17,2023-11-14,2023-11-14,Capital,Multi,...,2.0,0,SAO PAULO,8,3.339039,2.511007,2.850046,5.850046,1,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
437135,500725,126764327-1,RJ,Transportadora 2,2023-11-20 14:30:23,2023-11-24,2023-11-19,2023-11-19,Interior,Mono,...,7.0,0,CAMPOS DOS GOYTACAZES,14,1.604433,8.171042,4.775475,9.775475,1,5.0
437137,502182,126796645,RJ,Transportadora 2,2023-11-23 01:27:54,2023-11-27,2023-11-19,2023-11-19,Interior,Mono,...,8.0,0,CAMPOS DOS GOYTACAZES,1,4.061042,6.310741,2.371782,10.371782,1,8.0
438093,489718,127937390-1,SP,Transportadora 5,2023-12-06 11:23:44,2023-12-06,2023-11-28,2023-11-28,Capital,Multi,...,7.0,0,SAO PAULO,11,8.474815,1.243137,1.717951,9.717951,1,8.0
438095,500730,126590568,SP,Transportadora 5,2023-11-22 14:48:22,2023-11-22,2023-11-17,2023-11-17,Capital,Multi,...,4.0,0,SAO PAULO,14,5.616921,0.675023,1.291944,6.291944,1,5.0
